# CoT Generator — Komparasi & Pilih Pemenang

Bandingkan dua generator CoT yang sudah dinilai judge **Qwen2.5-7B (vLLM)** di pipeline masing-masing:

- **Gemma-2** (`cot_pipeline_kaggle_gemma.ipynb`)
- **DeepSeek-R1-Distill-Qwen-7B** (`cot_pipeline_kaggle.ipynb`)

Input = `correct.jsonl` dari kedua run (verdict judge sudah jadi, judge sama -> bisa langsung dibandingkan). **Tanpa GPU, tanpa re-judge.**

Metrik pemenang = **coverage**: jumlah soal unik (`id`) dengan >=1 solusi benar. Kalau pool soal beda, ranking pakai **intersection** (head-to-head fair).

Output: tabel komparasi + `compare_summary.json` + nama pemenang & path datanya. Tidak menyalin/membangun SFT — `sft/` run pemenang dipakai apa adanya di hilir.

## 0. Setup (clone repo buat import `src`)

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', URL, REPO])
sys.path.insert(0, REPO)
WORK = Path('/kaggle/working')
print('repo siap:', REPO)

## 1. Cari input (kedua `correct.jsonl`)

Auto-detect di `/kaggle/input` (label dari nama folder dataset). Kalau ambigu, isi `GEMMA_CORRECT` / `DEEPSEEK_CORRECT` manual.

In [ ]:
# Override manual (biarkan None buat auto-detect):
GEMMA_CORRECT    = None
DEEPSEEK_CORRECT = None

def _label(p):
    s = str(p).lower()
    if 'gemma' in s:
        return 'gemma'
    if 'deepseek' in s or 'r1' in s or 'qwen' in s:
        return 'deepseek'
    return None

RUNS = {}
if GEMMA_CORRECT:
    RUNS['gemma'] = Path(GEMMA_CORRECT)
if DEEPSEEK_CORRECT:
    RUNS['deepseek'] = Path(DEEPSEEK_CORRECT)

if len(RUNS) < 2:
    hits = sorted(Path('/kaggle/input').rglob('correct.jsonl'))
    print('correct.jsonl ketemu:')
    for h in hits:
        print(' -', h)
    for h in hits:
        lab = _label(h)
        if lab and lab not in RUNS:
            RUNS[lab] = h

assert len(RUNS) == 2, (
    f'Butuh 2 correct.jsonl (gemma+deepseek), ketemu: {dict((k, str(v)) for k, v in RUNS.items())}. '
    'Set GEMMA_CORRECT / DEEPSEEK_CORRECT manual.')
print('runs:', {k: str(v) for k, v in RUNS.items()})

## 2. Agregasi + komparasi + pemenang

- `coverage_raw` = soal unik terjawab benar.
- `coverage_common` = coverage di **intersection** pool yang sama-sama dicoba (butuh `candidates.jsonl`; kalau gak ada, acceptance rate = n/a).
- Pemenang = coverage tertinggi (intersection kalau pool beda), tie-break total solusi benar.

In [ ]:
import json
from src.cot_synthesis.utils import read_jsonl

def _pool_ids(correct_path):
    """Pool soal yang DICOBA (dari candidates.jsonl sebelah). None kalau tak ketemu."""
    cand = correct_path.parent / 'candidates.jsonl'
    if not cand.exists():
        hits = list(correct_path.parent.parent.rglob('candidates.jsonl'))
        cand = hits[0] if hits else None
    if cand and cand.exists():
        return {r['id'] for r in read_jsonl(cand)}
    return None

labs = list(RUNS)
metrics, solved, pools = {}, {}, {}
for lab, path in RUNS.items():
    rows = read_jsonl(path)
    sids = {r['id'] for r in rows}
    pool = _pool_ids(path)
    solved[lab], pools[lab] = sids, pool
    metrics[lab] = {
        'coverage_raw': len(sids),
        'total_correct': len(rows),
        'pool_size': len(pool) if pool else None,
        'acceptance_rate': (len(sids) / len(pool)) if pool else None,
        'correct_path': str(path),
    }

# Fairness: intersection pool yang dicoba kedua run.
if all(pools[l] for l in labs):
    common = pools[labs[0]] & pools[labs[1]]
    pools_equal = pools[labs[0]] == pools[labs[1]]
else:
    common = solved[labs[0]] | solved[labs[1]]
    pools_equal = None
    print('WARNING: candidates.jsonl tak lengkap -> pool yang dicoba tak diketahui; '
          'intersection pakai union id terlihat, acceptance rate n/a.')

if pools_equal is False:
    print('WARNING: pool soal kedua run BEDA -> ranking pakai intersection.')

for lab in labs:
    metrics[lab]['coverage_common'] = len(solved[lab] & common)

key = 'coverage_raw' if len(common) == 0 else 'coverage_common'
if len(common) == 0:
    print('WARNING: intersection KOSONG -> fallback ke coverage_raw.')

winner = max(labs, key=lambda l: (metrics[l][key], metrics[l]['total_correct'], l))

hdr = f"{'generator':10} {'cov_raw':>8} {'cov_common':>11} {'total_correct':>14} {'acc_rate':>9}"
print(hdr)
print('-' * len(hdr))
for lab in labs:
    m = metrics[lab]
    acc = f"{m['acceptance_rate']:.1%}" if m['acceptance_rate'] is not None else 'n/a'
    print(f"{lab:10} {m['coverage_raw']:>8} {m['coverage_common']:>11} {m['total_correct']:>14} {acc:>9}")
print()
print(f'WINNER: {winner}  (by {key})')
print(f"winner correct.jsonl: {metrics[winner]['correct_path']}")
print(f"-> pakai sft/ dari run '{winner}' untuk training SFT")

summary = {
    'generators': metrics,
    'common_pool_size': len(common),
    'pools_equal': pools_equal,
    'winner': winner,
    'winner_correct_path': metrics[winner]['correct_path'],
}
out = WORK / 'compare_summary.json'
out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('summary ->', out)

assert winner in labs and metrics[winner]['total_correct'] > 0